In [1]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [2]:
import torch
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

ZS_MODEL_1_NAME = "facebook/bart-large-mnli"

zero_shot_bart = pipeline(
    "zero-shot-classification",
    model=ZS_MODEL_1_NAME,
    device=0 if torch.cuda.is_available() else -1
)

print("Loaded:", ZS_MODEL_1_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded: facebook/bart-large-mnli


In [3]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("val.csv")


In [4]:
# Prompt variants
prompt_variants_bart = [
    "This comment is {} to the Reddit post question.",
    "This Reddit comment should be classified as {}.",
    "This comment answered the Reddit post question as {}."
]

# Candidate labels
candidate_labels_bart = [
    "a direct answer",
    "related but not an answer",
    "off-topic"
]

label_name_to_code = {
    "a direct answer": "direct",
    "related but not an answer": "related",
    "off-topic": "off-topic"
}


In [5]:
# Sample from training set for prompt selection
sample_df_bart = train_df.sample(n=min(100, len(train_df)), random_state=42).copy()

prompt_results_bart = []

for i, template in enumerate(prompt_variants_bart, start=1):
    preds = []

    for _, row in sample_df_bart.iterrows():
        post = str(row["Post"])
        comment = str(row["Comment"])

        post_short = post[:600]
        comment_short = comment[:600]

        text = f"Post: {post_short}\nComment: {comment_short}"

        result = zero_shot_bart(
            text,
            candidate_labels=candidate_labels_bart,
            hypothesis_template=template
        )

        pred_name = result["labels"][0]
        pred_code = label_name_to_code[pred_name]
        preds.append(pred_code)

    y_true = sample_df_bart["Label"].tolist()
    y_pred = preds

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    result_dict = {
        "prompt_number": i,
        "template": template,
        "accuracy": round(acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4)
    }

    prompt_results_bart.append(result_dict)
    print(result_dict)



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'prompt_number': 1, 'template': 'This comment is {} to the Reddit post question.', 'accuracy': 0.5, 'precision': 0.6471, 'recall': 0.5, 'f1': 0.5497}
{'prompt_number': 2, 'template': 'This Reddit comment should be classified as {}.', 'accuracy': 0.69, 'precision': 0.5983, 'recall': 0.69, 'f1': 0.6341}
{'prompt_number': 3, 'template': 'This comment answered the Reddit post question as {}.', 'accuracy': 0.67, 'precision': 0.6377, 'recall': 0.67, 'f1': 0.6474}


In [6]:
# Select best prompt
best_result_bart = max(prompt_results_bart, key=lambda x: x["f1"])
best_prompt_index_bart = best_result_bart["prompt_number"] - 1
BEST_PROMPT_ZS1 = prompt_variants_bart[best_prompt_index_bart]

print("\nBest BART prompt number:", best_result_bart["prompt_number"])
print("Best BART prompt metrics:", best_result_bart)
print("\nBest BART prompt:")
print(BEST_PROMPT_ZS1)

# Evaluate on validation set
eval_df = valid_df.copy()

preds = []

for _, row in eval_df.iterrows():
    post = str(row["Post"])
    comment = str(row["Comment"])

    text = f"Post: {post}\nComment: {comment}"
    text_short = text[:1500]

    result = zero_shot_bart(
        text_short,
        candidate_labels=candidate_labels_bart,
        hypothesis_template=BEST_PROMPT_ZS1
    )

    pred_name = result["labels"][0]
    pred_code = label_name_to_code[pred_name]
    preds.append(pred_code)

y_true = eval_df["Label"].tolist()
y_pred = preds

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\nValidation Results")
print("Accuracy:", round(acc, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

# Save predictions
results_df = eval_df.copy()
results_df["Predicted_Label"] = y_pred

results_df[["Post", "Comment", "Label", "Predicted_Label"]].head()


Best BART prompt number: 3
Best BART prompt metrics: {'prompt_number': 3, 'template': 'This comment answered the Reddit post question as {}.', 'accuracy': 0.67, 'precision': 0.6377, 'recall': 0.67, 'f1': 0.6474}

Best BART prompt:
This comment answered the Reddit post question as {}.

Validation Results
Accuracy: 0.6542
Precision: 0.621
Recall: 0.6542
F1: 0.6322

Classification Report:

              precision    recall  f1-score   support

      direct       0.76      0.85      0.80       164
   off-topic       0.29      0.33      0.31        18
     related       0.33      0.21      0.26        58

    accuracy                           0.65       240
   macro avg       0.46      0.46      0.45       240
weighted avg       0.62      0.65      0.63       240



,Post,Comment,Label,Predicted_Label
0,"""People who swore an oath to defend the consti...","""I'm a veteran and father of 2 mixed teenagers...",direct,direct
1,"""This teacher of ours lectured us on how diffi...","""Well, when I was learning coding (it was C++)...",related,related
2,"""People who swore an oath to defend the consti...","""I am disgusted. I served on active duty for o...",direct,direct
3,"""How much do you spend on groceries per month?...","""$300""",direct,direct
4,"""What has US politics taught you so far ?""","""This nation is being milked by capitalists un...",direct,direct
